In [10]:
import csv

SUBJECTS = [
    "Maths",
    "Physics",
    "Chemistry",
    "English",
    "Computer_Science"
]

MISSING_VALUES = {"", "na", "n/a", "-", "none", "null"}

MIN_MARK = 0
MAX_MARK = 100

MIN_ATTENDANCE = 0
MAX_ATTENDANCE = 100


def load_data(filename):
    with open(filename, newline="") as file:
        reader = csv.DictReader(file)
        data = list(reader)

    return data


def inspect_data(data):
    if len(data) == 0:
        return {
            "rows": 0,
            "columns": [],
            "missing": {}
        }

    columns = list(data[0].keys())
    missing_count = {}

    for column in columns:
        missing_count[column] = 0

    for row in data:
        for column in columns:
            value = row[column].strip().lower()

            if value in MISSING_VALUES:
                missing_count[column] += 1

    return {
        "rows": len(data),
        "columns": columns,
        "missing": missing_count
    }


def is_missing(value):
    return value.strip().lower() in MISSING_VALUES


def convert_to_number(value):
    if is_missing(value):
        return None

    try:
        return float(value)
    except ValueError:
        return None


def fix_text(row):
    row["name"] = " ".join(row["name"].split()).title()
    row["section"] = row["section"].strip().upper()

    class_text = row["class"].strip().lower()

    if "10" in class_text:
        row["class"] = "10"
    elif "11" in class_text:
        row["class"] = "11"
    elif "12" in class_text:
        row["class"] = "12"

    return row


def clean_data(data):

    cleaned_data = []
    student_ids = set()

    stats = {
        "duplicates": 0,
        "missing_marks": 0,
        "invalid_marks": 0,
        "missing_attendance": 0,
        "invalid_attendance": 0,
        "text_fixed": 0
    }

    for row in data:

        sid = row["student_id"]

        if sid in student_ids:
            stats["duplicates"] += 1
            continue

        student_ids.add(sid)

        valid = True
        marks = {}

        for subject in SUBJECTS:

            value = convert_to_number(row[subject])

            if value is None:
                stats["missing_marks"] += 1
                valid = False
                break

            if value < MIN_MARK or value > MAX_MARK:
                stats["invalid_marks"] += 1
                valid = False
                break

            marks[subject] = int(value)

        if not valid:
            continue

        attendance = convert_to_number(row["attendance_pct"])

        if attendance is None:
            stats["missing_attendance"] += 1
            continue

        if attendance < MIN_ATTENDANCE or attendance > MAX_ATTENDANCE:
            stats["invalid_attendance"] += 1
            continue

        old_values = (
            row["name"],
            row["class"],
            row["section"]
        )

        row = fix_text(row)

        new_values = (
            row["name"],
            row["class"],
            row["section"]
        )

        if old_values != new_values:
            stats["text_fixed"] += 1

        for subject in SUBJECTS:
            row[subject] = marks[subject]

        row["attendance_pct"] = attendance

        cleaned_data.append(row)

    return cleaned_data, stats


def save_clean_csv(data, filename):

    if len(data) == 0:
        return

    columns = list(data[0].keys())

    with open(filename, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        writer.writerows(data)


def main():

    raw_data = load_data("student_marks_raw.csv")

    structure = inspect_data(raw_data)

    clean_data_rows, statistics = clean_data(raw_data)

    save_clean_csv(
        clean_data_rows,
        "clean_student_marks.csv"
    )

    print("Data Cleaning Completed Successfully!")
    print("--------------------------------------")
    print("Raw Rows :", len(raw_data))
    print("Clean Rows :", len(clean_data_rows))
    print("Rows Removed :", len(raw_data) - len(clean_data_rows))
    print()

    print("Missing Values Per Column:")
    for column, count in structure["missing"].items():
        print(f"{column}: {count}")

    print("\nCleaning Summary:")
    print("Duplicate Rows Removed:", statistics["duplicates"])
    print("Rows with Missing Marks:", statistics["missing_marks"])
    print("Rows with Invalid Marks:", statistics["invalid_marks"])
    print("Rows with Missing Attendance:", statistics["missing_attendance"])
    print("Rows with Invalid Attendance:", statistics["invalid_attendance"])
    print("Rows with Text Fixed:", statistics["text_fixed"])

    print("\nOutput File Created:")
    print("clean_student_marks.csv")


if __name__ == "__main__":
    main()

Data Cleaning Completed Successfully!
--------------------------------------
Raw Rows : 203
Clean Rows : 117
Rows Removed : 86

Missing Values Per Column:
student_id: 0
name: 0
class: 0
section: 0
Maths: 10
Physics: 9
Chemistry: 19
English: 15
Computer_Science: 6
attendance_pct: 14

Cleaning Summary:
Duplicate Rows Removed: 3
Rows with Missing Marks: 47
Rows with Invalid Marks: 22
Rows with Missing Attendance: 11
Rows with Invalid Attendance: 3
Rows with Text Fixed: 104

Output File Created:
clean_student_marks.csv
